# Payment Fraud Detection - Demo Cleanup

This notebook resets the demo environment while preserving core infrastructure.

## What this notebook does:
- **Cleans up ML schemas**: Removes `FINSERV_FEATURE_STORE` and `FINSERV_MODEL_REGISTRY` schemas
- **Drops demo tables**: Clears `RAW_DATA` schema tables (ACTIVITIES, CHALLENGES, SUBSCRIPTIONS)
- **Removes ML artifacts**: Deletes feature entities, models, and stages created during demo
- **Preserves infrastructure**: Keeps role, warehouse, database, and PUBLIC schema intact

## What is preserved:
- ✅ `FINSERV_DEMO_ADMIN` role
- ✅ `STANDARD_WH_01_XS` warehouse  
- ✅ `FINSERV_FRAUD_DEMO` database
- ✅ `FINSERV_FRAUD_DEMO.PUBLIC` schema (untouched)
- ✅ `FINSERV_FRAUD_DEMO.RAW_DATA` schema structure (tables cleared, not dropped)

## Prerequisites:
- Connected to Snowflake with appropriate permissions
- Running in **Snowflake Notebooks** or with proper authentication


In [ ]:
# 🔗 Setup Connection
print("🔗 Connecting to Snowflake...")

# Import required libraries
import snowflake.snowpark as snowpark
from snowflake.ml.feature_store import FeatureStore, CreationMode

# Get Snowflake session (works in both Snowflake Notebooks and locally in Cursor)
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    print("✅ Running in Snowflake Notebooks")
except:
    from snowflake_connection import get_local_session
    session = get_local_session()
    print("✅ Running locally in Cursor")

print("✅ Connected to Snowflake")
print(f"🏢 Account: {session.get_current_account()}")
print(f"👤 User: {session.get_current_user()}")
print(f"🏗️ Warehouse: {session.get_current_warehouse()}")
print(f"🗃️ Database: {session.get_current_database()}")

# Switch to demo database
session.use_database("FINSERV_FRAUD_DEMO")
print(f"🎯 Working with database: {session.get_current_database()}")
print("="*60)


In [ ]:
# 🧹 Clean Up Feature Store Entities
print("🧹 Cleaning up Feature Store entities...")

try:
    # Try to connect to existing Feature Store and clean up entities
    fs = FeatureStore(
        session=session,
        database="FINSERV_FRAUD_DEMO",
        name="FINSERV_FEATURE_STORE",
        default_warehouse="STANDARD_WH_01_XS",
        creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
    )
    
    try:
        fs.delete_entity("customer")
        print("Deleted 'customer' entity from Feature Store")
    except Exception as e:
        print(f"Customer entity not found or already deleted: {str(e)}")
    
    print("✅ Feature Store entity cleanup completed")
    
except Exception as e:
    print(f"⚠️ Feature Store not found or inaccessible: {str(e)}")
    print("   This is normal if Feature Store was never created")

print("="*60)


In [ ]:
# 🗑️ Drop ML Schemas and Models
print("🗑️ Dropping ML schemas and models...")

# Drop ML schemas (CASCADE removes all objects within them)
schemas_to_drop = [
    "FINSERV_FEATURE_STORE",
    "FINSERV_MODEL_REGISTRY"
]

for schema in schemas_to_drop:
    try:
        session.sql(f"DROP SCHEMA IF EXISTS FINSERV_FRAUD_DEMO.{schema} CASCADE").collect()
        print(f"Dropped schema: {schema}")
    except Exception as e:
        print(f"Could not drop schema {schema}: {str(e)}")

models_to_drop = [
    "FRAUD_DETECTION_RF"
]

for model in models_to_drop:
    try:
        session.sql(f"DROP MODEL IF EXISTS FINSERV_FRAUD_DEMO.FINSERV_MODEL_REGISTRY.{model}").collect()
        print(f"Dropped model: {model}")
    except Exception as e:
        print(f"Could not drop model {model}: {str(e)}")

print("✅ ML schema cleanup completed")
print("="*60)


In [ ]:
# 🧽 Clear Demo Data Tables
print("🧽 Clearing demo data tables...")

# Switch to RAW_DATA schema
session.use_schema("RAW_DATA")

# Tables to clear (truncate, not drop - preserving structure)
tables_to_clear = [
    "TRANSACTIONS",
    "CUSTOMERS", 
    "MERCHANTS"
]

for table in tables_to_clear:
    try:
        # Check if table exists first
        result = session.sql(f"SHOW TABLES LIKE '{table}'").collect()
        if result:
            session.sql(f"TRUNCATE TABLE {table}").collect()
            print(f"✅ Cleared table: {table}")
        else:
            print(f"⚠️ Table {table} does not exist (skipping)")
    except Exception as e:
        print(f"⚠️ Could not clear table {table}: {str(e)}")

# Drop any temporary stages that might have been created
try:
    session.sql("SHOW STAGES").collect()
    # Drop any stages with 'TEMP' or 'SNOWPARK' in the name
    stages_result = session.sql("SHOW STAGES").collect()
    for stage in stages_result:
        stage_name = stage['name']
        if 'TEMP' in stage_name.upper() or 'SNOWPARK' in stage_name.upper():
            try:
                session.sql(f"DROP STAGE IF EXISTS {stage_name}").collect()
                print(f"✅ Dropped temporary stage: {stage_name}")
            except:
                pass
except Exception as e:
    print(f"⚠️ Could not check/drop temporary stages: {str(e)}")

print("✅ Demo data cleanup completed")
print("="*60)


In [ ]:
# ✅ Verify Cleanup and Show Preserved Infrastructure
print("✅ Verifying cleanup and showing preserved infrastructure...")

# Show what was preserved
print("\n🏗️ PRESERVED INFRASTRUCTURE:")
print("-" * 40)

# Check role
try:
    current_role = session.sql("SELECT CURRENT_ROLE()").collect()[0][0]
    print(f"✅ Role: {current_role}")
except:
    print("⚠️ Could not verify current role")

# Check warehouse
try:
    current_wh = session.get_current_warehouse()
    print(f"✅ Warehouse: {current_wh}")
except:
    print("⚠️ Could not verify current warehouse")

# Check database
try:
    current_db = session.get_current_database()
    print(f"✅ Database: {current_db}")
except:
    print("⚠️ Could not verify current database")

# Show schemas in database
try:
    schemas = session.sql("SHOW SCHEMAS").collect()
    print(f"\n📁 SCHEMAS IN {current_db}:")
    for schema in schemas:
        schema_name = schema['name']
        print(f"   • {schema_name}")
except:
    print("⚠️ Could not list schemas")

# Show tables in RAW_DATA (should be empty now)
try:
    session.use_schema("RAW_DATA")
    tables = session.sql("SHOW TABLES").collect()
    print(f"\n📊 TABLES IN RAW_DATA SCHEMA:")
    if tables:
        for table in tables:
            table_name = table['name']
            count_result = session.sql(f"SELECT COUNT(*) FROM {table_name}").collect()
            row_count = count_result[0][0]
            print(f"   • {table_name}: {row_count:,} rows")
    else:
        print("   (No tables found)")
except Exception as e:
    print(f"⚠️ Could not verify RAW_DATA tables: {str(e)}")

print(f"\n🎉 CLEANUP COMPLETED!")
print("=" * 60)
print("✅ Demo environment reset successfully")
print("✅ Core infrastructure preserved")
print("✅ Ready for fresh demo run")
print("\n💡 Next steps:")
print("   1. Run 01_env_setup.sql (or 01_execute.ipynb)")
print("   2. Run 02_generate_and_load_data_snowflake_native.ipynb")
print("   3. Run 03_finserv_fraud_demo_native.ipynb")
